In [ ]:
assert False


from singleCAM_IROS._pipeline_support import _handle_dirpaths

from typing import Any, NamedTuple

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt

from bloodmoon.mask import CodedMaskCamera, codedmask, decode
from bloodmoon.coords import shift2pos, angle2shift
from bloodmoon.optim import model_shadowgram
from bloodmoon.io import simulation_files

import darksun as ds
from darksun.types import Tag
from darksun.show import DSPlot

from var import sky_variance, sky_significance

basepath = '/mnt/d/PhD_AASS/Coding/Images_fits'

test = 'singleCAM_iros_testing'
_, true = ds.load_sky(f'{basepath}/{test}/COMPOSED_sky_SIMUL_CAM1A_CAM1B_TEST_singleCAM_iros_testing.fits')
_, iros = ds.load_sky(f'{basepath}/{test}/COMPOSED_OUTsky_IROS_CAM1A_CAM1B_TEST_singleCAM_iros_testing.fits')
log_camA, _ = ds.load_database(f'{basepath}/{test}/IROS_sources_database_TEST_{test}.fits')

maskpath = f'{basepath}/wfm_mask_summer2021.fits'
UPX, UPY = 5, 1
wfm = codedmask(maskpath, UPX, UPY)
ASPECT = 1 #(wfm.specs['mask_deltay'] / UPY) / (wfm.specs['mask_deltax'] / UPX)


save_to = '/mnt/d/PhD_AASS/Workshops/AASS Workshop 2025'
dsp = DSPlot(1, 1)

def plotting(
    arr: NDArray,
    save_to: str | None,
    tags: tuple[Tag] | None = None,
    show: bool = False,
    **kwargs: Any,
) -> None:
    fig, ax = dsp.config_subplots(ptype='image')
    ax.imshow(arr, origin='lower', **kwargs)
    if tags is not None:
        tags_ = (tags,) if isinstance(tags, Tag) else tags
        for (name, y, x) in tags_:
            ax.scatter(
                x, y, s=20, facecolors='None', edgecolors='white',
                linewidths=0.4, alpha=0.9,
            )
            ax.text(
                x + 125, y, name, color='white',
                fontsize=7, fontweight='bold',
            )
    ax.axis('off')
    if save_to:
        fig.savefig(save_to, pad_inches=0)
    if show:
        plt.show()
    plt.close()


TRUE_EXP = 0.8
TRUE_VMAX = 20

IROS_EXP = 0.8
IROS_VMAX = 20

SHOW = False
CMAP = 'plasma'

#true_ = np.pow(np.clip(true, a_min=1e-5, a_max=None), TRUE_EXP)
#gctrue = f'{save_to}/GC_preiros.png'
#plotting(
#    true_, gctrue, tags=None, show=SHOW,
#    aspect=ASPECT, vmin=0, vmax=TRUE_VMAX, cmap=CMAP,
#)


iros_ = np.pow(np.clip(iros, a_min=1e-5, a_max=None), IROS_EXP)

#ys, xs = np.unravel_index(np.argwhere(iros_ > 5)[0], shape=iros_.shape)
#iros_tag = tuple(
#    Tag(ID, y, x) for ID, y, x in zip(
#        log_camA.log['ID'], ys, xs,
#    )
#)

gciros = f'{save_to}/GC_afteriros.png'
plotting(
    iros_, gciros, tags=None, show=SHOW,
    aspect=ASPECT, vmin=0, vmax=IROS_VMAX, cmap=CMAP,
)

# Loading data...
# Loading completed!
# Loading data...
# Loading completed!
# Loading data...
# Loading completed!


In [ ]:
assert False


from typing import Any, NamedTuple

import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt

from bloodmoon.mask import CodedMaskCamera, codedmask, decode
from bloodmoon.coords import shift2pos, angle2shift
from bloodmoon.optim import model_shadowgram

from darksun.types import Tag
from darksun.show import DSPlot

from var import sky_variance, sky_significance


# setup mask and instr effects
basepath = '/mnt/d/PhD_AASS/Coding/Images_fits'
# basepath = '/home/edoardo/Datadisk/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations'
maskpath = f'{basepath}/wfm_mask_summer2021.fits'

UPX, UPY = 5, 1
wfm = codedmask(maskpath, UPX, UPY)
VIGNETTING = True
PSFY = False

ASPECT = (wfm.specs['mask_deltay'] / UPY) / (wfm.specs['mask_deltax'] / UPX)


save_to = '/mnt/d/PhD_AASS/Workshops/AASS Workshop 2025'
dsp = DSPlot(1, 1)

def plotting(
    arr: NDArray,
    save_to: str | None,
    tags: tuple[Tag] | None = None,
    show: bool = False,
    **kwargs: Any,
) -> None:
    fig, ax = dsp.config_subplots(ptype='image')
    ax.imshow(arr, origin='lower', **kwargs)
    if tags is not None:
        tags_ = (tags,) if isinstance(tags, Tag) else tags
        for (name, y, x) in tags_:
            ax.scatter(
                x, y, s=20, facecolors='None', edgecolors='white',
                linewidths=0.4, alpha=0.9,
            )
            ax.text(
                x + 125, y, name, color='white',
                fontsize=7, fontweight='bold',
            )
    ax.axis('off')
    if save_to:
        fig.savefig(save_to, pad_inches=0)
    if show:
        plt.show()
    plt.close()

class Candidate(NamedTuple):
    shift_x: float
    shift_y: float
    fluence: float

def gen_detector(
    candidates: tuple[Candidate],
    camera: CodedMaskCamera,
    vignetting: bool,
    psfy: bool,
) -> NDArray:
    """Generates detector image from retrieved candidates."""
    img = np.zeros(camera.shape_detector)
    for (sx, sy, f) in candidates:
        shadowgram = model_shadowgram(
            camera=camera,
            shift_x=sx,
            shift_y=sy,
            vignetting=vignetting,
            psfy=psfy,
        )
        img += (f * shadowgram)
    return img


sxs = (
    angle2shift(wfm, -20.0),
    angle2shift(wfm, 20.0),
    angle2shift(wfm, 20.0),
    angle2shift(wfm, -20.0),
)
sys = (
    angle2shift(wfm, -20.0),
    angle2shift(wfm, -20.0),
    angle2shift(wfm, 20.0),
    angle2shift(wfm, 20.0),
)
fluences = (10, 80, 30, 20)

candidates = tuple(
    Candidate(sx, sy, f) for sx, sy, f in zip(
        sxs, sys, fluences,
    )
)

# detector
detector = gen_detector(candidates, wfm, VIGNETTING, PSFY)
ds = f'{save_to}/detector_image_preiros.png'
#plotting(detector, ds, cmap='hot', aspect=ASPECT)

variance = sky_variance(wfm, detector)

# sky
sky = decode(wfm, detector)

# sky significance
n, m  = wfm.shape_sky
snr = sky_significance(sky, variance, ycut=int(1e-2 * n), xcut=int(1e-2 * m))

# after IROS
idx = np.argmax(fluences)
sx_iros, sy_iros, f_iros = sxs[idx], sys[idx], fluences[idx]
detector_iros = detector - f_iros * model_shadowgram(wfm, sx_iros, sy_iros, VIGNETTING, PSFY)
sky_iros = decode(wfm, detector_iros)
snr_iros = sky_significance(sky_iros, variance, ycut=int(1e-2 * n), xcut=int(1e-2 * m))


# PLOTs
SKY_EXP = 0.6
SKY_VMAX = 5

SNR_EXP = 0.6
SNR_VMAX = 5

SHOW = False
CMAP = 'plasma'

PRE_TAGS = Tag('S1', *shift2pos(wfm, sx_iros, sy_iros))
TAGS = tuple(
    Tag(f'S{idx}', *shift2pos(wfm, sx, sy))
    for (idx, sx), sy in zip(enumerate(sxs), sys)
)

# sky and significance
sky_ = np.pow(np.clip(sky, a_min=1e-5, a_max=None), SKY_EXP)
ss = f'{save_to}/sky_image_preiros.png'
plotting(
    sky_, ss, tags=PRE_TAGS, show=SHOW,
    aspect=ASPECT, vmin=0, vmax=SKY_VMAX, cmap=CMAP,
)

snr_ = np.pow(np.clip(snr, a_min=1e-5, a_max=None), SNR_EXP)
ssnr = None #f'{save_to}/skysnr_image_preiros.png'
plotting(
    snr_, ssnr, tags=TAGS, show=SHOW,
    aspect=ASPECT, vmin=0, vmax=SNR_VMAX, cmap=CMAP,
)

# sky and significance after IROS
sky_iros_ = np.pow(np.clip(sky_iros, a_min=1e-5, a_max=None), SKY_EXP)
ss = f'{save_to}/sky_image_afteriros.png'
plotting(
    sky_iros_, ss, tags=TAGS, show=SHOW,
    aspect=ASPECT, vmin=0, vmax=SKY_VMAX, cmap=CMAP,
)

snr_iros_ = np.pow(np.clip(snr_iros, a_min=1e-5, a_max=None), SNR_EXP)
ssnr = None #f'{save_to}/skysnr_image_afteriros.png'
plotting(
    snr_iros_, ssnr, tags=TAGS, show=SHOW,
    aspect=ASPECT, vmin=0, vmax=SNR_VMAX, cmap=CMAP,
)

In [ ]:
assert False


import numpy as np
import matplotlib.pyplot as plt

from bloodmoon.mask import codedmask, decode
from bloodmoon.coords import angle2shift
from bloodmoon.optim import model_shadowgram

from darksun.show import DSPlot


# setup mask and instr effects
basepath = '/mnt/d/PhD_AASS/Coding/Images_fits'
# basepath = '/home/edoardo/Datadisk/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations'
maskpath = f'{basepath}/wfm_mask_summer2021.fits'

UPX, UPY = 5, 1
wfm = codedmask(maskpath, UPX, UPY)
VIGNETTING = True
PSFY = False

ASPECT = (wfm.specs['mask_deltay'] / UPY) / (wfm.specs['mask_deltax'] / UPX)


save_to = '/mnt/d/PhD_AASS/Workshops/AASS Workshop 2025'
dsp = DSPlot(1, 1)

def plotting(arr, cmap, save_to, **kwargs):
    fig, ax = dsp.config_subplots(ptype='image')
    ax.imshow(arr, cmap=cmap, origin='lower', **kwargs)
    ax.axis('off')
    if save_to:
        fig.savefig(save_to, pad_inches=0)
    plt.close()

# camera mask
ms = f'{save_to}/wfm_mask.png'
plotting(wfm.mask, 'hot', ms, aspect=ASPECT)


# detector image
shifty, shiftx = (
    angle2shift(wfm, 20.0),
    angle2shift(wfm, 20.0),
)
detector = model_shadowgram(wfm, shiftx, shifty, VIGNETTING, PSFY)
ds = f'{save_to}/detector_image.png'
plotting(detector, 'hot', ds, aspect=ASPECT)


# sky image
fluence = 100
sky = decode(wfm, detector) * fluence

sky_ = np.pow(np.clip(sky, a_min=1e-5, a_max=None), 0.85)
ss = f'{save_to}/sky_image.png'
plotting(sky_, 'hot', ss, aspect=ASPECT, vmin=0)


# sky significance
from var import sky_variance, sky_significance

n, m  = wfm.shape_sky

variance = sky_variance(wfm, detector)
snr = sky_significance(sky, variance, ycut=int(1e-2 * n), xcut=int(1e-2 * m))

snr_ = np.pow(np.clip(snr, a_min=1e-5, a_max=None), 0.95)
ssnr = f'{save_to}/skysnr_image.png'
plotting(snr_, 'plasma', ssnr, aspect=ASPECT, vmin=0, vmax=50)

In [ ]:
assert False

import numpy as np
from matplotlib.colors import ListedColormap as lc
import matplotlib.pyplot as plt
from scipy.signal import correlate
from sympy import isprime, primerange

import darksun as ds

save_to = '/mnt/d/PhD_AASS/Workshops/AASS Workshop 2025'
dsp = ds.show.DSPlot(1, 1)
#CMAP = lc(["DodgerBlue", "DeepSkyBlue"])
CMAP = lc(["MediumVioletRed", "Orange"])

def img_plotting(arr, cmap, save_to, show=False):
    fig, ax = dsp.config_subplots(ptype='image')
    ax.imshow(arr, cmap=cmap, origin='lower')
    ax.axis('off')
    fig.savefig(save_to, pad_inches=0)
    if show:
        plt.show()
    plt.close()

# URA mask
class URAMaskPattern:
    """Generates a 2D URA pattern for a coded mask camera."""
    
    def __init__(self, rank: int):
        
        self._check_rank(rank)
        self.rank = rank

        self.prime_pair = self._get_prime_pair(rank)
        C_r_i, C_s_j = self._get_pattern_root()
        self.basic_pattern = self._get_basic_pattern(C_r_i, C_s_j)
        self.basic_decoder = self._get_decoder()
    

    def _check_rank(self, rank):
        if rank < 0:
            raise ValueError(f"rank must be >= 0, got rank = {rank} instead.")
    

    def _get_prime_pair(self, rank) -> tuple[int, int]:

        assert rank >= 0

        lim = int(1e4)
        primes = list(primerange(2, lim))
        p1, this_rank = primes[0], -1

        for p2 in primes[1:]:
            if (p2 - p1) == 2:
                this_rank += 1
                if this_rank == rank:
                    return p2, p1
            p1 = p2

        raise ValueError(f"Could not find prime pairs in the range [2, {lim}] for rank = {rank}.")
    

    def _get_pattern_root(self):

        r, s = self.prime_pair
        assert isprime(r)
        assert isprime(s)
        assert r - s == 2

        C_r_i = np.zeros(r) - 1
        C_s_j = np.zeros(s) - 1

        for x in range(1, r):
            C_r_i[x**2 % r] = 1
        for y in range(1, s):
            C_s_j[y**2 % s] = 1
        
        return C_r_i, C_s_j


    def _get_basic_pattern(self, C_r_i, C_s_j):

        A = np.zeros(self.prime_pair)

        for i in range(self.prime_pair[0]):
            for j in range(self.prime_pair[1]):

                if i == 0: A[i,j] = 0
                elif j == 0: A[i,j] = 1
                elif C_r_i[i]*C_s_j[j] == 1: A[i,j] = 1
                else: A[i,j] = 0
        
        return A
    

    def _get_decoder(self):

        G = 2*self.basic_pattern - 1
        G /= self.basic_pattern.sum()

        return G

ura = URAMaskPattern(3)


# mask pattern
#img_plotting(ura.basic_pattern, CMAP, f'{save_to}/ura_mask.png')


# PSF
pad_n, pad_m = map(
    lambda x: (x - 1) // 2,
    (ura.basic_pattern.shape)
)
psf = correlate(
    np.pad(ura.basic_decoder, (pad_n, pad_m), mode='wrap'),
    ura.basic_pattern,
    mode='valid',
)
u, v = psf.shape
dmapx = ds.map4plot(
    arrs=psf[u // 2, :],
    title="",
    xlabel='x [px]',
    ylabel='counts [ph]',
    x=np.arange(v + 1) - v // 2 - 0.5,
    style='stairs',
    color='OrangeRed',
    ylim=(-0.1, None)
)
ds.plot(
    dmapx,
    save_to=f'{save_to}/ura_PSF_slice.png'
)
#img_plotting(psf, CMAP, f'{save_to}/ura_PSF.png')


# pattern projection
n, m = ura.prime_pair
sky = np.zeros((n, m))

offy, offx = 3, 4
sky[n // 2 + offy, m // 2 + offx] = 1
encoding = correlate(
    np.pad(ura.basic_pattern, (pad_n, pad_m), mode='wrap'),
    sky,
    mode='valid',
)
#img_plotting(encoding, CMAP, f'{save_to}/ura_proj.png')

In [ ]:
assert False

import numpy as np
from matplotlib.colors import ListedColormap as lc
import matplotlib.pyplot as plt
from scipy.signal import correlate

import darksun as ds
from darksun.show import DSPlot

save_to = '/mnt/d/PhD_AASS/Workshops/AASS Workshop 2025'
dsp = DSPlot(1, 1)
CMAP = lc(["DarkBlue", "DeepSkyBlue"])

def plotting(arr, cmap, save_to, show=False):
    fig, ax = dsp.config_subplots(ptype='image')
    ax.imshow(arr, cmap=cmap, origin='lower')
    ax.axis('off')
    fig.savefig(save_to, pad_inches=0)
    if show:
        plt.show()
    plt.close()

# chessboard mask
width = 8
seed1 = [
    1 if n % 2 == 0 else 0 for n in range(width)
]
seed2 = seed2 = seed1[1:] + [1]
chessmask = np.array(
    [
        seed1 if n % 2 == 0 else seed2 for n in range(width)
    ]
)
#plotting(chessmask, CMAP, f'{save_to}/chessboard_mask.png')

# PSF
chessmask_decoder = (2 * chessmask - 1) / chessmask.sum()
pad_n, pad_m = map(
    lambda x: (x - 1) // 2,
    (chessmask.shape)
)
psf = correlate(
    np.pad(chessmask_decoder, (pad_n, pad_m), mode='wrap'),
    chessmask,
    mode='valid',
)
u, v = psf.shape
dmapx = ds.map4plot(
    arrs=psf[u // 2, :],
    title="",
    xlabel='x [px]',
    ylabel='counts [ph]',
    x=np.arange(v + 1) - v // 2 - 0.5,
    style='stairs',
    color='OrangeRed',
    ylim=(-1.1, 1.1)
)
ds.plot(
    dmapx,
    save_to=f'{save_to}/chessboard_PSF_slice.png'
)
#plotting(psf, CMAP, f'{save_to}/chessboard_PSF.png')


# pattern projection
n, m = chessmask.shape
sky = np.zeros((n, m))

off = 3
sky[n // 2 + off, m // 2 + off] = 1
encoding = correlate(
    np.pad(chessmask, (pad_n, pad_m), mode='wrap'),
    sky,
    mode='valid',
)
#plotting(encoding, CMAP, f'{save_to}/chessboard_proj.png')